**Task2:** *JSON Schema classification and info extraction*

1. This notebook demonstrates structured information extraction from user chats.
2. Using Groq API with function calling, we extract details into a JSON schema.

In [ ]:
import os
import json
from jsonschema import validate

# importing all the necessary libraries and setting up the Groq API key

os.environ["GROQ_API_KEY"] = "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"

# Initializing the client object in openAI instance

from openai import OpenAI
client = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")

In [17]:
# Well defined JSON schema for validation

schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": ["integer", "null"]},
        "location": {"type": ["string", "null"]},
        "email": {"type": ["string", "null"]},
        "phone": {"type": ["string", "null"]},
    },
    "required": ["name", "age", "location", "email", "phone"]
}

In [18]:
# Now define the function schema for structured extraction
# This tells the model how to extract user details in a structured way.

functions = [
    {
        "name": "extract_user_info",                               # Name of the function
        "description": "Extract user details from conversation",   # What the function does
        "parameters": schema                                       # The JSON schema defined in previous cell
    }
]


# This function sends the user chats to the LLM for structured extraction of the information
# suggested in the schema.

def extract_info(chat_text):
  response = client.chat.completions.create(
      model="llama-3.1-8b-instant",
      messages=[{"role": "system", "content": """You are an information extraction engine.
            Extract the following fields from the user input:
            - name (string)
            - age (integer)
            - location (string)
            - email (string)
            - phone (string)

            Always return only a valid JSON object that matches this schema exactly.
            If some info is missing, use string 'unclear'.
            Do not leave fields blank or null.
            """},
          {"role": "user", "content": chat_text}],
      functions=functions,
      function_call={"name": "extract_user_info"}
  )

  # Now parsing the JSON response

  data = json.loads(response.choices[0].message.function_call.arguments)
  return data

In [19]:
# Now this function ensures the program does not fails after extraction fails (if it does).

def safe_extract_info(chat_text):     # chat_text = raw conversation text from the user.
    try:
        data = extract_info(chat_text)
    except Exception as e:
        print("Extraction failed:", e)
        data = {
            "name": None,
            "age": None,
            "location": None,
            "email": None,
            "phone": None
        }
    return data

    # returns dict: extracted details in JSON format.

In [20]:
# This is the demonstration block , where we will test our program by passing various inputs

examples = [
    "Hi, I’m Rahul Sharma, 24 years old, living in Delhi. My email is rahul@gmail.com and phone is 9876543210",
    "My name is Priya Verma, age 30. I stay in Mumbai, email priya.v@gmail.com, phone 9123456789",
    "Hello, I’m Ankit, 22, from Bangalore. Contact: ankit22@yahoo.com, phone 9000000001",
    "Hi, My name is Ashish. I'm 24 years old currently staying in Chandigarh. My email is ashish@gmail.com and the phone number is 1234567890",
    "Hi, My name is Harish, 14 years old currently staying in Kolkata. My email is abc@gmail.com"

]

for text in examples:
    print("Input:", text)
    print("Extracted:", safe_extract_info(text))
    print("-" * 50)

Input: Hi, I’m Rahul Sharma, 24 years old, living in Delhi. My email is rahul@gmail.com and phone is 9876543210
Extracted: {'age': 24, 'email': 'rahul@gmail.com', 'location': 'Delhi', 'name': 'Rahul Sharma', 'phone': '9876543210'}
--------------------------------------------------
Input: My name is Priya Verma, age 30. I stay in Mumbai, email priya.v@gmail.com, phone 9123456789
Extracted: {'age': 30, 'email': 'priya.v@gmail.com', 'location': 'Mumbai', 'name': 'Priya Verma', 'phone': '9123456789'}
--------------------------------------------------
Input: Hello, I’m Ankit, 22, from Bangalore. Contact: ankit22@yahoo.com, phone 9000000001
Extraction failed: Error code: 400 - {'error': {'message': 'tool call validation failed: parameters for tool extract_user_info did not match schema: errors: [`/age`: expected integer or null, but got string]', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=extract_user_info> {"name": "Ankit", "age": "22", "loca